## **1. Objetivo General del Rol (Negocio)**

Actuar como consultora especializada en analítica para **StreamView Analytics** con el fin de **comprender el comportamiento de los usuarios y apoyar la toma de decisiones estratégicas** en cuatro frentes clave:

- **Retención** de clientes.

- Nivel de interacción (**engagement**).

- Preferencias de **consumo de contenido.**

- **Experiencia** de usuario.

## **2. Objetivos Específicos del Proyecto (Lo que deben lograr)**

- **Integrar y preparar datos:** Unificar las distintas fuentes corporativas (usuarios, suscripciones, reproducciones, dispositivos, calificaciones e interacciones) identificando las variables críticas para el análisis.

- **Explorar y detectar patrones:** Realizar un análisis exploratorio visual para encontrar tendencias, relaciones clave y hallazgos orientados al negocio.

- **Diseñar y justificar visualizaciones:** Crear gráficos pertinentes justificando la elección de encodificación visual (canales, marcas, percepción) según la audiencia.

- **Construir una herramienta analítica (Dashboard):** Implementar un panel interactivo (ej. Power BI, Tableau, Dash, Streamlit) que incluya KPIs principales, navegación fluida y filtros para la toma de decisiones.

- **Comunicar con Data Storytelling:** Estructurar una narrativa visual comprensible que guíe al espectador desde el dato hasta la conclusión, adaptando el lenguaje al perfil de la audiencia.

- **Evaluar críticamente el trabajo:** Identificar objetivamente las fortalezas, limitaciones y oportunidades de mejora de la solución implementada.

- **Formular recomendaciones accionables:** Proponer decisiones concretas basadas estrictamente en la evidencia de los datos analizados.

- **Garantizar la reproducibilidad técnica:** Entregar el código, datos y estructura de carpetas (data/, notebooks/, dashboard/, src/, README.md) listos para que cualquier persona pueda replicar el proyecto sin errores.

### Limpieza basica y cruzamiento de datasets

In [3]:
import pandas as pd

# Cargamos cada fuente en un DataFrame separado para poder revisar y limpiar
# las peliculas y las series con el mismo procedimiento.
df_movies_raw = pd.read_csv('../data/netflix_movies_detailed_up_to_2025.csv')
df_tv_shows_raw = pd.read_csv('../data/netflix_tv_shows_detailed_up_to_2025.csv')


def limpiar_catalogo(df, fuente):
    """Aplica una limpieza basica y agrega campos comunes al catalogo."""
    df = df.copy()

    # Estandarizamos los nombres de columnas para evitar diferencias de escritura.
    df.columns = (
        df.columns.str.strip()
        .str.lower()
        .str.replace(' ', '_', regex=False)
    )

    # Convertimos a texto antes de limpiar para tolerar columnas completamente vacias.
    columnas_texto = df.select_dtypes(include=['object', 'string']).columns
    for columna in columnas_texto:
        df[columna] = df[columna].astype('string').str.strip()
        df[columna] = df[columna].replace({'': pd.NA, 'nan': pd.NA, 'None': pd.NA})

    # Eliminamos columnas que no tienen ningun dato en esta fuente.
    columnas_vacias = df.columns[df.isna().all()].tolist()
    df = df.drop(columns=columnas_vacias)
    columnas_vacias_por_fuente[fuente] = columnas_vacias

    # Convertimos las columnas de fecha y numericas para facilitar el analisis.
    if 'date_added' in df.columns:
        df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')

    columnas_numericas = [
        'show_id', 'release_year', 'rating', 'popularity',
        'vote_count', 'vote_average', 'budget', 'revenue'
    ]
    for columna in columnas_numericas:
        if columna in df.columns:
            df[columna] = pd.to_numeric(df[columna], errors='coerce')

    # Conservamos el origen y creamos una llave unica para el catalogo combinado.
    df['source_dataset'] = fuente
    df['content_type'] = df['type'].astype('string').str.strip().str.lower()
    df['content_key'] = fuente + '_' + df['show_id'].astype('Int64').astype(str)

    # Separamos duration en un valor numerico y una unidad (minutos o temporadas).
    if 'duration' in df.columns:
        duration_texto = df['duration'].astype('string')
        partes_duration = duration_texto.str.extract(
            r'(?P<duration_value>\d+)\s*(?P<duration_unit>.*)'
        )
        df['duration_value'] = pd.to_numeric(partes_duration['duration_value'], errors='coerce')
        df['duration_unit'] = partes_duration['duration_unit'].str.strip().replace('', pd.NA)

    # Un show_id repetido dentro de una misma fuente representa un registro duplicado.
    df = df.drop_duplicates(subset='show_id', keep='first').reset_index(drop=True)
    return df


# Este diccionario documenta las columnas completamente vacias eliminadas por fuente.
columnas_vacias_por_fuente = {}

# Limpiamos ambas fuentes con la misma funcion para que sean comparables.
df_movies = limpiar_catalogo(df_movies_raw, 'movies')
df_tv_shows = limpiar_catalogo(df_tv_shows_raw, 'tv_shows')

# Como las columnas adicionales no son iguales en ambas fuentes, concat permite
# conservar toda la informacion y completa con NaN donde una fuente no tiene un campo.
df_catalogo = pd.concat(
    [df_movies, df_tv_shows],
    ignore_index=True,
    sort=False
)

# Eliminamos cualquier columna que haya quedado completamente vacia despues de unir.
columnas_vacias_catalogo = df_catalogo.columns[df_catalogo.isna().all()].tolist()
df_catalogo = df_catalogo.drop(columns=columnas_vacias_catalogo)

# Validaciones simples para comprobar que el cruce genero un marco coherente.
assert len(df_catalogo) == len(df_movies) + len(df_tv_shows)
assert df_catalogo['content_key'].is_unique
assert not df_catalogo.isna().all().any()

print(f'Peliculas limpias: {df_movies.shape[0]:,} filas')
print(f'Series limpias: {df_tv_shows.shape[0]:,} filas')
print(f'Catalogo combinado: {df_catalogo.shape[0]:,} filas y {df_catalogo.shape[1]:,} columnas')
print('\nColumnas completamente vacias eliminadas por fuente:')
print(columnas_vacias_por_fuente)
print(f'Columnas completamente vacias eliminadas despues de unir: {columnas_vacias_catalogo}')
print('\nRegistros por tipo:')
print(df_catalogo['content_type'].value_counts(dropna=False))

# Este reporte permite revisar los faltantes parciales antes del analisis.
reporte_faltantes = (
    df_catalogo.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename('porcentaje_faltante')
    .to_frame()
)
reporte_faltantes.head(10)

Peliculas limpias: 16,000 filas
Series limpias: 15,991 filas
Catalogo combinado: 31,991 filas y 23 columnas

Columnas completamente vacias eliminadas por fuente:
{'movies': ['duration'], 'tv_shows': []}
Columnas completamente vacias eliminadas despues de unir: []

Registros por tipo:
content_type
movie      16000
tv show    15991
Name: count, dtype: Int64


,porcentaje_faltante
duration_value,50.014066
duration,50.014066
duration_unit,50.014066
budget,49.985934
revenue,49.985934
director,34.672252
description,10.431059
country,7.067613
cast,4.251196
genres,3.372824


In [4]:
# CONFIRMACION: la limpieza y el cruce del catalogo se actualizaron correctamente.
assert 'df_catalogo' in globals()
assert not df_catalogo.empty
assert not df_catalogo.isna().all().any()
assert df_catalogo['content_key'].is_unique

print('ACTUALIZACION CONFIRMADA: df_catalogo esta listo para el analisis.')
print(f'Filas disponibles: {len(df_catalogo):,}')
print(f'Columnas disponibles: {df_catalogo.shape[1]:,}')

ACTUALIZACION CONFIRMADA: df_catalogo esta listo para el analisis.
Filas disponibles: 31,991
Columnas disponibles: 23


In [5]:
# Vista previa del dataset limpio y combinado.
df_catalogo.head(10)

,show_id,type,title,director,cast,country,date_added,release_year,rating,genres,...,vote_count,vote_average,budget,revenue,source_dataset,content_type,content_key,duration,duration_value,duration_unit
0,10192,Movie,Shrek Forever After,Mike Mitchell,"Mike Myers, Eddie Murphy, Cameron Diaz, Antoni...",United States of America,2010-05-16,2010,6.380,"Comedy, Adventure, Fantasy, Animation, Family",...,7449,6.380,165000000.0,752600867.0,movies,movie,movies_10192,<NA>,<NA>,<NA>
1,27205,Movie,Inception,Christopher Nolan,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W...","United Kingdom, United States of America",2010-07-15,2010,8.369,"Action, Science Fiction, Adventure",...,37119,8.369,160000000.0,839030630.0,movies,movie,movies_27205,<NA>,<NA>,<NA>
2,12444,Movie,Harry Potter and the Deathly Hallows: Part 1,David Yates,"Daniel Radcliffe, Emma Watson, Rupert Grint, T...","United Kingdom, United States of America",2010-11-17,2010,7.744,"Adventure, Fantasy",...,19327,7.744,250000000.0,954305868.0,movies,movie,movies_12444,<NA>,<NA>,<NA>
3,38757,Movie,Tangled,"Byron Howard, Nathan Greno","Mandy Moore, Zachary Levi, Donna Murphy, Ron P...",United States of America,2010-11-24,2010,7.600,"Animation, Family, Adventure",...,11638,7.600,260000000.0,592461732.0,movies,movie,movies_38757,<NA>,<NA>,<NA>
4,10191,Movie,How to Train Your Dragon,"Chris Sanders, Dean DeBlois","Jay Baruchel, Gerard Butler, Craig Ferguson, A...",United States of America,2010-03-18,2010,7.800,"Fantasy, Adventure, Animation, Family",...,13259,7.800,165000000.0,494879471.0,movies,movie,movies_10191,<NA>,<NA>,<NA>
5,11324,Movie,Shutter Island,Martin Scorsese,"Leonardo DiCaprio, Mark Ruffalo, Ben Kingsley,...",United States of America,2010-02-14,2010,8.200,"Drama, Thriller, Mystery",...,24282,8.200,80000000.0,294804195.0,movies,movie,movies_11324,<NA>,<NA>,<NA>
6,38575,Movie,The Karate Kid,Harald Zwart,"Jaden Smith, Jackie Chan, Taraji P. Henson, We...","China, Hong Kong, United States of America",2010-06-10,2010,6.500,"Action, Adventure, Drama, Family",...,6082,6.500,40000000.0,359126022.0,movies,movie,movies_38575,<NA>,<NA>,<NA>
7,10138,Movie,Iron Man 2,Jon Favreau,"Robert Downey Jr., Gwyneth Paltrow, Don Cheadl...",United States of America,2010-04-28,2010,6.800,"Adventure, Action, Science Fiction",...,21222,6.800,200000000.0,623933331.0,movies,movie,movies_10138,<NA>,<NA>,<NA>
8,38365,Movie,Grown Ups,Dennis Dugan,"Adam Sandler, Kevin James, Chris Rock, David S...",United States of America,2010-06-24,2010,6.394,Comedy,...,6212,6.394,80000000.0,271430189.0,movies,movie,movies_38365,<NA>,<NA>,<NA>
9,48650,Movie,Room in Rome,Julio Medem,"Elena Anaya, Natasha Yarovenko, Enrico Lo Vers...","France, Spain",2010-05-07,2010,6.416,"Drama, Romance",...,748,6.416,0.0,844281.0,movies,movie,movies_48650,<NA>,<NA>,<NA>


In [6]:
# Eliminación de columnas que no aportan al análisis.
columnas_a_eliminar = [
    'director', 'cast', 'country', 'date_added', 'language', 'description',
    'vote_count', 'budget', 'revenue', 'source_dataset', 'content_type',
    'content_key', 'duration_value', 'duration_unit', 'duration'
]

# Solo elimina columnas que existen para evitar errores por nombres mal escritos.
df_catalogo = df_catalogo.drop(
    columns=[col for col in columnas_a_eliminar if col in df_catalogo.columns]
)


In [7]:
df_catalogo

,show_id,type,title,release_year,rating,genres,popularity,vote_average
0,10192,Movie,Shrek Forever After,2010,6.380,"Comedy, Adventure, Fantasy, Animation, Family",203.893,6.380
1,27205,Movie,Inception,2010,8.369,"Action, Science Fiction, Adventure",156.242,8.369
2,12444,Movie,Harry Potter and the Deathly Hallows: Part 1,2010,7.744,"Adventure, Fantasy",121.191,7.744
3,38757,Movie,Tangled,2010,7.600,"Animation, Family, Adventure",111.762,7.600
4,10191,Movie,How to Train Your Dragon,2010,7.800,"Fantasy, Adventure, Animation, Family",110.044,7.800
...,...,...,...,...,...,...,...,...
31986,284892,TV Show,Sammelanam,2025,0.000,"Comedy, Drama",3.236,0.000
31987,277665,TV Show,Anne Shirley,2025,0.000,"Animation, Drama, Family",3.558,0.000
31988,284972,TV Show,Le onde del passato,2025,10.000,Drama,2.913,10.000
31989,284613,TV Show,AI히치하이커,2025,0.000,<NA>,2.787,0.000


In [8]:
# Separa cada genero en una lista por fila
df_catalogo['genres_list'] = (
    df_catalogo['genres']
    .fillna('')
    .astype(str)
    .str.strip()
    .str.replace(r"[\[\]'\"]", '', regex=True)
    .str.split(',')
)

df_catalogo['genres_list'] = df_catalogo['genres_list'].apply(
    lambda x: [g.strip() for g in x if g.strip() != '']
)

In [9]:
# Crea columnas dinamicas: genre_1, genre_2, genre_3, ...
genres_expandidos = (
    df_catalogo['genres_list']
    .apply(pd.Series)
    .rename(columns=lambda x: f'genre_{x + 1}')
)

df_catalogo = pd.concat([df_catalogo, genres_expandidos], axis=1)

In [10]:
df_catalogo

,show_id,type,title,release_year,rating,genres,popularity,vote_average,genres_list,genre_1,genre_2,genre_3,genre_4,genre_5,genre_6,genre_7,genre_8
0,10192,Movie,Shrek Forever After,2010,6.380,"Comedy, Adventure, Fantasy, Animation, Family",203.893,6.380,"[Comedy, Adventure, Fantasy, Animation, Family]",Comedy,Adventure,Fantasy,Animation,Family,NaN,NaN,NaN
1,27205,Movie,Inception,2010,8.369,"Action, Science Fiction, Adventure",156.242,8.369,"[Action, Science Fiction, Adventure]",Action,Science Fiction,Adventure,NaN,NaN,NaN,NaN,NaN
2,12444,Movie,Harry Potter and the Deathly Hallows: Part 1,2010,7.744,"Adventure, Fantasy",121.191,7.744,"[Adventure, Fantasy]",Adventure,Fantasy,NaN,NaN,NaN,NaN,NaN,NaN
3,38757,Movie,Tangled,2010,7.600,"Animation, Family, Adventure",111.762,7.600,"[Animation, Family, Adventure]",Animation,Family,Adventure,NaN,NaN,NaN,NaN,NaN
4,10191,Movie,How to Train Your Dragon,2010,7.800,"Fantasy, Adventure, Animation, Family",110.044,7.800,"[Fantasy, Adventure, Animation, Family]",Fantasy,Adventure,Animation,Family,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31986,284892,TV Show,Sammelanam,2025,0.000,"Comedy, Drama",3.236,0.000,"[Comedy, Drama]",Comedy,Drama,NaN,NaN,NaN,NaN,NaN,NaN
31987,277665,TV Show,Anne Shirley,2025,0.000,"Animation, Drama, Family",3.558,0.000,"[Animation, Drama, Family]",Animation,Drama,Family,NaN,NaN,NaN,NaN,NaN
31988,284972,TV Show,Le onde del passato,2025,10.000,Drama,2.913,10.000,[Drama],Drama,NaN,NaN,NaN,NaN,NaN,NaN,NaN
31989,284613,TV Show,AI히치하이커,2025,0.000,<NA>,2.787,0.000,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
